# EXPECT: phần thực hiện của Long

Chỉ train/dev. Notebook kiểm tra và hiển thị đầu ra thật đã lưu. Đặt RUN_TRAINING=True để gọi lại pipeline dev; model/cache hợp lệ được dùng lại. Không có cell đánh giá test.

In [1]:
from pathlib import Path
import os, sys, json
ROOT = Path.cwd()
if not (ROOT / 'run_dev.py').exists():
    raise RuntimeError('Mở notebook từ thư mục Explainable_GEC_Long')
sys.path.insert(0, str(ROOT))
import bootstrap
from gec.safety import install_development_guard
install_development_guard()
import pandas as pd
OUT = ROOT / 'outputs/dev_run_001'
print('Python:', sys.version.split()[0])
print('Chế độ dev: test.json/test.pkl bị chặn')

Python: 3.12.14
Chế độ dev: test.json/test.pkl bị chặn


In [2]:
RUN_TRAINING = False
if RUN_TRAINING:
    import subprocess
    subprocess.run([sys.executable, '-u', 'run_dev.py', '--stage', 'all'], cwd=ROOT, check=True)
manifest = json.loads((OUT / 'run_manifest.json').read_text(encoding='utf-8'))
assert manifest['status'] == 'complete'
assert manifest['splits_read'] == ['train', 'dev']
assert not manifest['test_scored']
print('CLI run:', manifest['status'], manifest['completed_at'])

CLI run: complete 2026-10-22T20:52:41+07:00


In [3]:
from gec.data import load_development
train, dev, summaries = load_development(ROOT / 'data/json')
print(pd.DataFrame([{k:v for k,v in s.items() if k in ['split','records','tokens','SVA','VFORM','NNUM']} for s in summaries]).to_string(index=False))

split  records  tokens  SVA  VFORM  NNUM
train    14610  416571 1005   1190  1668
  dev     2367   68382   74     69   120


In [4]:
metrics = pd.read_csv(OUT / 'metrics_dev.csv')
print(metrics[metrics['label'].eq('MICRO')][['system','scoring','TP','FP','FN','P','R','F0.5']].to_string(index=False))

                system scoring  TP  FP  FN        P        R     F0.5
                 All_O   typed   0   0 263 0.000000 0.000000 0.000000
                 All_O untyped   0   0 263 0.000000 0.000000 0.000000
                   POS   typed  36 222 227 0.139535 0.136882 0.138996
                   POS untyped  37 221 226 0.143411 0.140684 0.142857
               LR_base   typed  38 112 225 0.253333 0.144487 0.220162
               LR_base untyped  40 110 223 0.266667 0.152091 0.231750
LR_reference_C3_tau065   typed  28  65 235 0.301075 0.106464 0.220472
LR_reference_C3_tau065 untyped  30  63 233 0.322581 0.114068 0.236220
             LR_parser   typed  27  55 236 0.329268 0.102662 0.228426
             LR_parser untyped  28  54 235 0.341463 0.106464 0.236887
                   kNN   typed  10  79 253 0.112360 0.038023 0.080775
                   kNN untyped  11  78 252 0.123596 0.041825 0.088853


In [5]:
grid = pd.read_csv(OUT / 'tuning_grid.csv')
assert len(grid) == 228
print(pd.read_csv(OUT / 'lr_best_per_C.csv')[['system','C','tau','F0.5','selected_C']].to_string(index=False))
print('Grid rows:', len(grid))

   system   C  tau     F0.5  selected_C
  LR_base 0.1 0.30 0.215054       False
  LR_base 0.3 0.30 0.207769       False
  LR_base 1.0 0.45 0.220162        True
  LR_base 3.0 0.65 0.220472       False
LR_parser 0.1 0.35 0.228426        True
LR_parser 0.3 0.35 0.228967       False
LR_parser 1.0 0.60 0.224525       False
LR_parser 3.0 0.50 0.210872       False
Grid rows: 228


In [6]:
classes = metrics[metrics['system'].isin(['LR_base','LR_parser']) & metrics['label'].ne('MICRO')]
print(classes[['system','label','TP','FP','FN','P','R','F0.5']].to_string(index=False))
cases = pd.read_csv(OUT / 'parser_four_cases.csv')
print(cases[['report_case_name','token','subject_text','reference_C3_tau065_prediction','base_prediction','parser_prediction']].to_string(index=False))

   system label  TP  FP  FN        P        R     F0.5
  LR_base   SVA  12  41  62 0.226415 0.162162 0.209790
  LR_base VFORM   7  17  62 0.291667 0.101449 0.212121
  LR_base  NNUM  19  54 101 0.260274 0.158333 0.230583
LR_parser   SVA  12  16  62 0.428571 0.162162 0.322581
LR_parser VFORM   5  15  64 0.250000 0.072464 0.167785
LR_parser  NNUM  10  24 110 0.294118 0.083333 0.195312
report_case_name   token subject_text reference_C3_tau065_prediction base_prediction parser_prediction
            FP02 realize       people                            SVA             SVA               SVA
            FP03      is     learning                            SVA             SVA               SVA
            FN01      is      reasons                              O               O                 O
            FN04      is       <NONE>                              O             SVA                 O


In [7]:
for person in ['Loan','Yen']:
    review = pd.read_csv(OUT / 'baseline_ref' / f'review_{person}.csv', keep_default_na=False)
    done = review['review_label_A_B_C'].str.strip().ne('').sum()
    print(person, 'completed:', done, '/', len(review))
print('Phiếu để người đọc độc lập điền; chưa có precision rà soát.')

Loan completed: 0 / 63
Yen completed: 0 / 63
Phiếu để người đọc độc lập điền; chưa có precision rà soát.


In [8]:
verification = json.loads((OUT / 'verification.json').read_text(encoding='utf-8'))
assert verification['status'] == 'passed'
from gec.io_utils import sha256
locked = json.loads((OUT / 'config_locked.json').read_text(encoding='utf-8'))
for entry in locked['bundles'].values():
    assert sha256(ROOT / entry['path']) == entry['sha256']
print('Independent verification:', verification['status'])
print('Model checksums: matched')
print('Frozen state:', locked['status'], '; team review pending:', locked['team_review_pending'])
print('Test scored:', verification['test_scored'])

Independent verification: passed
Model checksums: matched
Frozen state: dev_selection_frozen ; team review pending: True
Test scored: False


Kết luận trên dev: SVA F0.5 tăng nhờ giảm FP, recall giữ nguyên. VFORM/NNUM giảm ở cấu hình parser được chọn. Chưa suy ra chất lượng trên test hoặc độ tin cậy phản hồi cho người học. Xem LONG_HANDOFF.md để đọc đầy đủ giới hạn và cách bàn giao.